In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [3]:
df = spark.read.json("cwiczenia/RTA/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [4]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [5]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [6]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [8]:
from pyspark.sql.functions import min as _min, max as _max, sum as _sum

df_stats = df.groupBy("category").agg(
    _sum("amount").alias("total_amount"),
    _min("amount").alias("min_amount"),
    _max("amount").alias("max_amount")
).orderBy("category")

df_stats.show()

+-----------+------------------+----------+----------+
|   category|      total_amount|min_amount|max_amount|
+-----------+------------------+----------+----------+
|elektronika|1520770.6899999995|       9.0|    9999.0|
|    książki| 851382.0799999991|       5.0|   9107.25|
|     odzież| 849877.5500000007|       5.0|   9696.63|
|    żywność| 789514.4300000003|       5.0|   6916.92|
+-----------+------------------+----------+----------+



In [9]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [10]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [12]:
from pyspark.sql.functions import window, count, sum as _sum, col

df_with_time = df.withColumn("timestamp", col("timestamp").cast("timestamp"))

df_windows = df_with_time.groupBy(
    window("timestamp", "30 minutes"), 
    "store"
).agg(
    count("tx_id").alias("transaction_count"),
    _sum("amount").alias("total_amount")
).orderBy("window", "store")

df_windows.show(20, truncate=False)

+------------------------------------------+--------+-----------------+------------------+
|window                                    |store   |transaction_count|total_amount      |
+------------------------------------------+--------+-----------------+------------------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252              |93391.22000000002 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289              |117786.4199999999 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275              |88441.58000000003 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296              |111540.5899999999 |
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514              |209187.85000000012|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532              |223541.41000000006|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490              |182435.05999999994|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502              |215587.1699999999 |

In [13]:
from pyspark.sql.functions import window, sum as _sum, desc, col

df_krakow = df.filter(col("store") == "Kraków") \
              .withColumn("timestamp", col("timestamp").cast("timestamp"))

df_top_hour = df_krakow.groupBy(
    window("timestamp", "1 hour")
).agg(
    _sum("amount").alias("total_revenue")
)

df_result = df_top_hour.orderBy(desc("total_revenue"))

df_result.show(truncate=False)

+------------------------------------------+------------------+
|window                                    |total_revenue     |
+------------------------------------------+------------------+
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|483309.85999999975|
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|341327.82999999996|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|201259.25999999995|
+------------------------------------------+------------------+



In [14]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [17]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# bo tumbling to okna rozlaczne, a w sliding okna na siebie nachodza 
# (wiec np godzina 8:45 nalezy do dwoch okien jednoczesnie - 8:00 do 9:00 oraz 8:30 do 9:30)

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [18]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") to agregacja globalna dla każdego sklepu, a groupBy(window(...), "store") to agregacja w czasie

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2 okna (okno 8:30-9:30 i okno 9:00-10:00)

In [20]:
from pyspark.sql.functions import window, sum as _sum, asc, col

df_gdansk = df.filter(col("store") == "Gdańsk") \
              .withColumn("timestamp", col("timestamp").cast("timestamp"))

df_avg_gdansk = df_gdansk.groupBy(
    window("timestamp", "1 hour")
).agg(
    _sum("amount").alias("avg_amount")
)

df_result = df_avg_gdansk.orderBy(asc("avg_amount"))

df_result.show(truncate=False)

+------------------------------------------+------------------+
|window                                    |avg_amount        |
+------------------------------------------+------------------+
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|230407.70000000007|
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|302579.07000000007|
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|488279.5799999997 |
+------------------------------------------+------------------+



In [21]:
from pyspark.sql.functions import window, count, col

df_time = df.withColumn("timestamp", col("timestamp").cast("timestamp"))

result_0930 = df_time.groupBy(
    window("timestamp", "30 minutes"), 
    "category"
).agg(
    count("tx_id").alias("liczba_transakcji")
).filter(
    (col("window.start") == "2026-04-12 09:00:00") & 
    (col("window.end") == "2026-04-12 09:30:00")
).orderBy("liczba_transakcji", ascending=False)

result_0930.select("category", "liczba_transakcji").show()

+-----------+-----------------+
|   category|liczba_transakcji|
+-----------+-----------------+
|    książki|              622|
|elektronika|              611|
|     odzież|              605|
|    żywność|              567|
+-----------+-----------------+



In [22]:
from pyspark.sql.functions import window, count, col, desc

df_time = df.withColumn("timestamp", col("timestamp").cast("timestamp"))

peaks_15min = df_time.groupBy(
    window("timestamp", "15 minutes")
).agg(
    count("tx_id").alias("liczba_transakcji")
).orderBy(desc("liczba_transakcji"))

peaks_15min.show(10, truncate=False)

+------------------------------------------+-----------------+
|window                                    |liczba_transakcji|
+------------------------------------------+-----------------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234             |
|{2026-04-12 09:00:00, 2026-04-12 09:15:00}|1171             |
|{2026-04-12 09:30:00, 2026-04-12 09:45:00}|1156             |
|{2026-04-12 08:45:00, 2026-04-12 09:00:00}|1139             |
|{2026-04-12 09:45:00, 2026-04-12 10:00:00}|1100             |
|{2026-04-12 08:30:00, 2026-04-12 08:45:00}|899              |
|{2026-04-12 10:00:00, 2026-04-12 10:15:00}|858              |
|{2026-04-12 08:15:00, 2026-04-12 08:30:00}|644              |
|{2026-04-12 10:15:00, 2026-04-12 10:30:00}|582              |
|{2026-04-12 08:00:00, 2026-04-12 08:15:00}|468              |
+------------------------------------------+-----------------+
only showing top 10 rows

